# Part 2: NeRF 3D Reconstruction

This notebook orchestrates Part 2 training and rendering using helper modules:
- `rendering.py`
- `nerf_model.py`
- `train_part2.py`
- `part2_utils.py`

Start with a smoke run, then scale to full training.

In [1]:
from pathlib import Path
import numpy as np
import torch

In [2]:
from part2_utils import ensure_dir, plot_training_curves, save_depth_png, save_rgb_png, set_seed
from train_part2 import build_part2_data, render_test_trajectory, train_nerf_part2

In [ ]:
# Core configuration (mid run profile)
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_path": "lego_200x200.npz",
    "output_dir": "images/output/part2",
    "hidden_dim": 256,
    "n_layers": 8,
    "pos_freqs": 10,
    "dir_freqs": 4,
    "n_coarse": 32,
    "n_fine": 32,
    "batch_rays": 1024,
    "n_steps": 1000,
    "eval_every": 50,
    "chunk_size": 4096,
    "lr": 5e-4,
    "near_override": 2.0,
    "far_override": 6.0,
}

set_seed(CFG["seed"])
ensure_dir(CFG["output_dir"])
CFG

{'seed': 42,
 'device': 'cpu',
 'data_path': 'lego_200x200.npz',
 'output_dir': 'images/output/part2',
 'hidden_dim': 256,
 'n_layers': 8,
 'pos_freqs': 10,
 'dir_freqs': 4,
 'n_coarse': 32,
 'n_fine': 32,
 'batch_rays': 1024,
 'n_steps': 500,
 'eval_every': 50,
 'chunk_size': 4096,
 'lr': 0.0005,
 'near_override': 2.0,
 'far_override': 6.0}

In [9]:
# Quick data sanity check (no training)
data = build_part2_data(CFG["data_path"], device=CFG["device"])
print("K shape:", tuple(data["k"].shape))
print("Train rays:", len(data["train_dataset"]))
print("Val images:", tuple(data["val_images"].shape))
print("Test poses:", tuple(data["test_c2ws"].shape))

K shape: (3, 3)
Train rays: 4000000
Val images: (10, 200, 200, 3)
Test poses: (60, 4, 4)


In [10]:
# Smoke training run (increase n_steps and batch_rays later)
results = train_nerf_part2(
    data_path=CFG["data_path"],
    output_dir=CFG["output_dir"],
    device=CFG["device"],
    seed=CFG["seed"],
    hidden_dim=CFG["hidden_dim"],
    n_layers=CFG["n_layers"],
    pos_freqs=CFG["pos_freqs"],
    dir_freqs=CFG["dir_freqs"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    n_steps=CFG["n_steps"],
    batch_rays=CFG["batch_rays"],
    lr=CFG["lr"],
    eval_every=CFG["eval_every"],
    chunk_size=CFG["chunk_size"],
    near_override=CFG["near_override"],
    far_override=CFG["far_override"],
)
results["metrics"]

[train_nerf_part2] start device=cpu steps=500 batch_rays=1024 coarse=32 fine=32 near=2.000 far=6.000 eval_every=50
[train_nerf_part2] step 1/500 loss=0.198607 coarse=0.079104 fine=0.190696 iter=4.783s rays_per_sec=214
[train_nerf_part2] step 25/500 loss=0.076268 coarse=0.064019 fine=0.069866 iter=4.205s rays_per_sec=244
[train_nerf_part2] step 50/500 loss=0.064441 coarse=0.063347 fine=0.058106 iter=4.384s rays_per_sec=234
[train_nerf_part2] eval step 50/500 val_psnr=12.072dB best=12.072dB (new best, saved checkpoint) eval=59.70s
[train_nerf_part2] step 75/500 loss=0.064719 coarse=0.061261 fine=0.058593 iter=4.408s rays_per_sec=232
[train_nerf_part2] step 100/500 loss=0.061712 coarse=0.063455 fine=0.055366 iter=4.096s rays_per_sec=250
[train_nerf_part2] eval step 100/500 val_psnr=13.860dB best=13.860dB (new best, saved checkpoint) eval=58.31s
[train_nerf_part2] step 125/500 loss=0.031601 coarse=0.035737 fine=0.028028 iter=4.106s rays_per_sec=249
[train_nerf_part2] step 150/500 loss=0.02

{'best_psnr': 19.909602403640747,
 'best_step': 500,
 'final_loss': 0.009746618568897247,
 'near': 2.0,
 'far': 6.0,
 'n_steps': 500,
 'batch_rays': 1024,
 'n_coarse': 32,
 'n_fine': 32,
 'val_psnr_hist': [12.071906328201294,
  13.860418796539307,
  16.97177290916443,
  17.82161235809326,
  18.379809856414795,
  18.872662782669067,
  19.20232057571411,
  19.343563318252563,
  19.517406225204468,
  19.909602403640747],
 'eval_steps': [50, 100, 150, 200, 250, 300, 350, 400, 450, 500],
 'total_seconds': 2774.542336300001,
 'avg_seconds_per_step': 5.549084672600002,
 'log_every': 25}

In [11]:
# Save training curves
curve_dir = Path(CFG["output_dir"]) / "curves"
plot_training_curves(
    loss_hist=results["loss_hist"],
    eval_steps=results["eval_steps"],
    val_psnr_hist=results["val_psnr_hist"],
    output_dir=curve_dir,
)
print("Saved curves to", curve_dir)

Saved curves to images\output\part2\curves


In [12]:
# Render 60 test views to NPY (RGB + depth)
models = results["models"]
data = results["data"]
image_hw = (data["val_images"].shape[1], data["val_images"].shape[2])

render_test_trajectory(
    model_coarse=models["coarse"],
    model_fine=models["fine"],
    k=data["k"],
    test_c2ws=data["test_c2ws"],
    image_hw=image_hw,
    output_dir=CFG["output_dir"],
    near=results["metrics"]["near"],
    far=results["metrics"]["far"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    chunk_size=CFG["chunk_size"],
    device=CFG["device"],
)
print("Saved test trajectory npy files.")

KeyboardInterrupt: 

In [13]:
# Optional: convert NPY outputs to PNG files
rgb_npy_dir = Path(CFG["output_dir"]) / "test_rgb_npy"
depth_npy_dir = Path(CFG["output_dir"]) / "test_depth_npy"
rgb_png_dir = Path(CFG["output_dir"]) / "test_renders"
depth_png_dir = Path(CFG["output_dir"]) / "depth_maps"

rgb_png_dir.mkdir(parents=True, exist_ok=True)
depth_png_dir.mkdir(parents=True, exist_ok=True)

for npy_path in sorted(rgb_npy_dir.glob("view_*.npy")):
    img = np.load(npy_path)
    save_rgb_png(img, rgb_png_dir / f"{npy_path.stem}.png")

for npy_path in sorted(depth_npy_dir.glob("view_*.npy")):
    dep = np.load(npy_path)
    save_depth_png(dep, depth_png_dir / f"{npy_path.stem}.png")

print("Saved PNG outputs:", rgb_png_dir, depth_png_dir)

Saved PNG outputs: images\output\part2\test_renders images\output\part2\depth_maps


In [14]:
# Launch Viser server for camera/ray/sample visualization
import sys
import subprocess
from pathlib import Path

viser_cmd = [
    sys.executable,
    "visualize_viser.py",
    "--data_path", CFG["data_path"],
    "--near", str(CFG["near_override"]),
    "--far", str(CFG["far_override"]),
    "--num_samples_along_ray", "64",
    "--num_rays", "300",
    "--num_cameras", "1",
    "--camera_start_idx", "0",
    "--device", CFG["device"],
    "--port", "8080",
]

viser_proc = subprocess.Popen(viser_cmd, cwd=str(Path.cwd()))
print(f"Viser started with PID={viser_proc.pid}")
print("Open http://localhost:8080")

Viser started with PID=33440
Open http://localhost:8080


In [15]:
# Stop Viser server launched from this notebook
try:
    viser_proc.terminate()
    viser_proc.wait(timeout=5)
    print("Viser stopped.")
except NameError:
    print("No running viser_proc found in this notebook session.")
except Exception as e:
    print("Could not stop cleanly:", e)

Viser stopped.
